# insurance-fairness

**Proxy discrimination auditing for UK insurance pricing models.**

This notebook shows the core workflow in under two minutes:
generate synthetic motor data, detect which rating factors act as
proxies for a protected characteristic, and read the RAG status table.

The finding that motivates this library: Citizens Advice (2022) estimated
a £213m/year ethnicity penalty in UK motor insurance driven by postcode
acting as an ethnicity proxy. Proxy R-squared detects this mechanically.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-fairness/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-fairness catboost polars scikit-learn scipy

## Synthetic data

We build 5,000 synthetic motor policies. The key design choice: postcode
is strongly correlated with `ethnicity_proxy` (ONS % non-white at LSOA
level). Vehicle age and NCD years are independent of it.

This replicates the Citizens Advice (2022) finding structure at small scale.

In [ ]:
import numpy as np
import polars as pl

rng = np.random.default_rng(2024)
n = 5_000

# Protected characteristic: ONS LSOA % non-white (continuous, 0-1)
ethnicity_proxy = rng.beta(2, 8, n)

# Postcode: 20 bands, sorted by ethnicity proxy — strong proxy
postcode_pct = np.argsort(np.argsort(ethnicity_proxy)) / n
postcode = np.array([f"PC{np.digitize(p, np.linspace(0,1,21))-1:02d}" for p in postcode_pct])

# Other factors: independent of ethnicity proxy
vehicle_age = rng.integers(1, 11, n)
ncd_years   = rng.integers(0, 6, n)
exposure    = rng.uniform(0.5, 1.0, n)

df = pl.DataFrame({
    "postcode":        postcode,
    "vehicle_age":     vehicle_age,
    "ncd_years":       ncd_years,
    "ethnicity_proxy": ethnicity_proxy,
    "exposure":        exposure,
})
print(df.shape)
df.head()

## Proxy detection

`proxy_r2_scores` fits a small CatBoost model for each rating factor,
predicting the protected characteristic from that factor alone.
High R-squared means the factor carries information about ethnicity.

In [ ]:
from insurance_fairness import proxy_r2_scores, mutual_information_scores

factors = ["postcode", "vehicle_age", "ncd_years"]

r2   = proxy_r2_scores(df, "ethnicity_proxy", factors, exposure_col="exposure",
                       catboost_iterations=100)
mi   = mutual_information_scores(df, "ethnicity_proxy", factors, exposure_col="exposure")

result = pl.DataFrame({
    "factor":       factors,
    "proxy_r2":     [round(r2[f], 4) for f in factors],
    "mutual_info":  [round(mi[f], 4) for f in factors],
    "rag":          ["RED" if r2[f] > 0.1 else "AMBER" if r2[f] > 0.05 else "GREEN" for f in factors],
}).sort("proxy_r2", descending=True)

print(result)

## What you should see

| factor | proxy_r2 | rag |
|--------|----------|-----|
| postcode | ~0.55 | RED |
| vehicle_age | ~0.00 | GREEN |
| ncd_years | ~0.00 | GREEN |

Postcode has a very high proxy R-squared because we constructed it
to be nearly perfectly correlated with `ethnicity_proxy`.
Vehicle age and NCD years score near zero — they carry no information
about the protected characteristic.

A proxy R-squared above 0.10 is the audit threshold for further
investigation. This is not an FCA-prescribed number, but it is
conservative enough to catch the kind of proxy relationship
documented in the Citizens Advice report.

## Combined audit with detect_proxies

`detect_proxies` runs R-squared, mutual information, and partial
Spearman correlation together and returns a ranked `ProxyDetectionResult`.

In [ ]:
from insurance_fairness.proxy_detection import detect_proxies

audit = detect_proxies(
    df,
    protected_col="ethnicity_proxy",
    factor_cols=factors,
    exposure_col="exposure",
    catboost_iterations=100,
)

print(f"Flagged factors: {audit.flagged_factors}")
print()
print(audit.to_polars())

## Next steps

This notebook covers proxy detection only. The full library also provides:

- **`FairnessAudit`** — end-to-end audit of a fitted CatBoost pricing model,
  including calibration by group, SHAP attribution, and FCA-ready Markdown report
- **`insurance_fairness.optimal_transport`** — discrimination-free pricing
  via Lindholm marginalisation and Wasserstein barycenter correction
- **`insurance_fairness.diagnostics`** — D_proxy scalar, Shapley attribution,
  per-policyholder vulnerability scores
- **Pareto optimisation** (`pip install insurance-fairness[pareto]`) — NSGA-II
  for the fairness–accuracy trade-off

**GitHub:** https://github.com/burning-cost/insurance-fairness  
**PyPI:** https://pypi.org/project/insurance-fairness/

The full demo notebook (`notebooks/fairness_audit_demo.py`) walks through
50,000-policy data with a fitted CatBoost model and FCA reporting.